In [2]:
# Problem Statement
# Identify charges for insurance based on the metrics provided.
# After looking into the dataset its numerical data. 
# Both input and outputs are available.
# Output label is numerical.
###################################################
# Domain - MACHINE LEARNING
# Supervised Learning
# Regression
###################################################

In [2]:
import pandas as pd

In [3]:
dataset = pd.read_csv("insurance_pre.csv")
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [4]:
# Dataset has 1338 Rows X 6 Columns
# We have to convert the following nominal fields to numerical using one-hot encoding.
# The nominal fields are smoker, sex.

In [4]:
# Preprocessing
dataset = pd.get_dummies(dataset, dtype="int64", drop_first=True)
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [5]:
#  Display Columns
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [6]:
# Identify independent variables
independent = dataset[["age", "bmi", "children", "sex_male", "smoker_yes"]]
independent

,age,bmi,children,sex_male,smoker_yes
0,19,27.900,0,0,1
1,18,33.770,1,1,0
2,28,33.000,3,1,0
3,33,22.705,0,1,0
4,32,28.880,0,1,0
...,...,...,...,...,...
1333,50,30.970,3,1,0
1334,18,31.920,0,0,0
1335,18,36.850,0,0,0
1336,21,25.800,0,0,0


In [7]:
# Identify dependent variable
dependent = dataset[["charges"]]
dependent

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520
...,...
1333,10600.54830
1334,2205.98080
1335,1629.83350
1336,2007.94500


In [8]:
# Split training and test set
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(independent, dependent, test_size = 0.30, random_state = 0)

In [9]:
# --------------------------------------------------
# Feature Scaling (Required for SVR)
### StandardScaler — Quick Notes
# * **StandardScaler** standardizes features so they have approximately **mean = 0** and **standard deviation = 1**.
# * Formula: **`z = (x - mean) / standard deviation`**
# * `fit()` → learns the **mean and standard deviation** from the training data.
# * `transform()` → applies the learned scaling to the data.
# * `fit_transform()` → performs both `fit()` and `transform()` together.
# * Use `fit_transform()` on **training data**.
# * Use only `transform()` on **test data** to avoid **data leakage**.
# * Scaling is especially important for **SVR, KNN, SVM, PCA, and neural networks**.
# * Scaling is generally **not required for Decision Trees, Random Forest, AdaBoost, XGBoost, and LightGBM**.
# --------------------------------------------------
from sklearn.preprocessing import StandardScaler
# Create the scaler
scaler = StandardScaler()
# Fit on training data and transform
x_train = scaler.fit_transform(x_train)
# Transform the test data using the same scaler
x_test = scaler.transform(x_test)

In [14]:
# Import GridSearchCV to automatically test different hyperparameter combinations
# using cross-validation and identify the best combination
from sklearn.model_selection import GridSearchCV

# Import DecisionTreeRegressor for performing Decision Tree Regression
from sklearn.tree import DecisionTreeRegressor

# Define the hyperparameters and the values that GridSearchCV should test
param_grid = {
    # Define different criteria used to measure the quality of a split
    # 'mse' and 'mae' are outdated names
    # mse => squared_error
    # mae => absolute_error
    'criterion': ['squared_error', 'absolute_error', 'friedman_mse'],

    # Define how many features should be considered when looking for the best split
    # None => consider all features
    # sqrt => consider sqrt(number of features)
    # log2 => consider log2(number of features)
    'max_features': [None, 'sqrt', 'log2'],

    # Define how the tree should choose the split
    # best => choose the best split
    # random => choose the best split among randomly selected splits
    'splitter': ['best', 'random']
}

# Create the GridSearchCV object
# DecisionTreeRegressor() is the model we want to tune
# param_grid contains all hyperparameter combinations to test
# refit=True means the best model will be trained again using the complete training data
# verbose=3 displays detailed progress information
# n_jobs=-1 uses all available CPU cores to speed up the search
grid = GridSearchCV(
    DecisionTreeRegressor(),
    param_grid,
    refit=True,
    verbose=3,
    n_jobs=-1
)

# Start the hyperparameter search using the training data
# GridSearchCV will train and evaluate the Decision Tree for every combination
grid.fit(x_train, y_train)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


,estimator,DecisionTreeRegressor()
,param_grid,"{'criterion': ['squared_error', 'absolute_error', ...], 'max_features': [None, 'sqrt', ...], 'splitter': ['best', 'random']}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'friedman_mse'


In [15]:
# Get all results generated by GridSearchCV
# cv_results_ contains information about every hyperparameter combination tested,
# including parameters, training scores, validation scores, fit time, and ranking
re = grid.cv_results_

# Convert the GridSearchCV results dictionary into a pandas DataFrame
# This makes the results easier to read, filter, sort, and analyze
table = pd.DataFrame.from_dict(re)

# Display the complete results table
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.011975,0.005886,0.006951,0.002825,squared_error,None,best,"{'criterion': 'squared_error', 'max_features':...",0.743449,0.545890,0.762977,0.640311,0.676523,0.673830,0.077838,3
1,0.008701,0.003720,0.008460,0.006799,squared_error,None,random,"{'criterion': 'squared_error', 'max_features':...",0.679167,0.635384,0.629247,0.535328,0.619432,0.619712,0.046887,10
2,0.008466,0.004526,0.006733,0.003969,squared_error,sqrt,best,"{'criterion': 'squared_error', 'max_features':...",0.662418,0.539310,0.655570,0.616528,0.634547,0.621675,0.044238,8
3,0.008120,0.004416,0.002607,0.000874,squared_error,sqrt,random,"{'criterion': 'squared_error', 'max_features':...",0.756973,0.645677,0.657736,0.463092,0.608646,0.626425,0.095276,7
4,0.006624,0.001598,0.002942,0.000639,squared_error,log2,best,"{'criterion': 'squared_error', 'max_features':...",0.698537,0.596523,0.619896,0.617962,0.510502,0.608684,0.060121,13
5,0.003762,0.000296,0.002107,0.000310,squared_error,log2,random,"{'criterion': 'squared_error', 'max_features':...",0.555268,0.556154,0.561708,0.586192,0.620652,0.575995,0.025002,16
6,0.050139,0.008756,0.002566,0.001325,absolute_error,None,best,"{'criterion': 'absolute_error', 'max_features'...",0.741031,0.594970,0.609882,0.542753,0.669242,0.631576,0.067970,6
7,0.033781,0.010723,0.002240,0.000362,absolute_error,None,random,"{'criterion': 'absolute_error', 'max_features'...",0.673739,0.613102,0.621764,0.612004,0.701617,0.644445,0.036541,4
8,0.028647,0.008409,0.002709,0.000420,absolute_error,sqrt,best,"{'criterion': 'absolute_error', 'max_features'...",0.643954,0.447884,0.616213,0.661843,0.735736,0.621126,0.095238,9
9,0.021763,0.013183,0.001927,0.000390,absolute_error,sqrt,random,"{'criterion': 'absolute_error', 'max_features'...",0.717730,0.691497,0.437796,0.498542,0.627808,0.594675,0.109058,15


In [16]:
# Import r2_score to evaluate how well the model predicts the target values
from sklearn.metrics import r2_score

# Use the best model found by GridSearchCV to make predictions on the test dataset
# grid.predict() automatically uses grid.best_estimator_
grid_predictions = grid.predict(x_test)

# Calculate the R2 score by comparing the actual test values (y_test)
# with the predictions made by the best model
r_score = r2_score(y_test, grid_predictions)

# Print the best hyperparameter combination found by GridSearchCV
# along with its R2 score on the unseen test dataset
print("The R2 value for best params {}: ".format(grid.best_params_), r_score)

The R2 value for best params {'criterion': 'friedman_mse', 'max_features': None, 'splitter': 'random'}:  0.7102039612559319


In [17]:
# Get all the results generated by GridSearchCV
# cv_results_ contains the parameters tested, CV scores, rankings,
# fitting time, and other information for every combination
re = grid.cv_results_

# Use the best model found by GridSearchCV to predict the test data
# grid.predict() automatically uses the best estimator after GridSearchCV
grid_predictions = grid.predict(x_test)

# Display the predictions made for each row in the test dataset
grid_predictions

array([10065.413   ,  8930.93455 , 46151.1245  , 13143.86485 ,
        9264.797   , 21984.47061 ,  2007.945   , 10848.1343  ,
        7418.522   ,  5253.524   ,  5385.3379  , 11150.78    ,
        7345.7266  ,  4571.41305 , 35147.52848 , 11741.726   ,
       12142.5786  ,  3292.52985 ,  6203.90175 , 34838.873   ,
       22218.1149  , 11881.9696  ,  9625.92    , 24535.69855 ,
        1826.843   ,  3875.7341  ,  3161.454   , 15828.82173 ,
        3757.8448  ,  8027.968   , 15828.82173 , 48673.5588  ,
       12981.3457  , 20781.48892 , 16115.3045  ,  3554.203   ,
        8334.5896  , 38711.      , 40941.2854  ,  1880.07    ,
        5266.3656  ,  2866.091   , 19539.243   , 49577.6624  ,
       40419.0191  ,  3579.8287  , 11741.726   ,  7160.094   ,
        4719.52405 , 12032.326   ,  2166.732   ,  3481.868   ,
       21978.6769  , 46661.4424  , 11856.4115  , 19673.33573 ,
        2497.0383  ,  8442.667   ,  7441.501   , 12913.9924  ,
        1712.227   , 46130.5265  , 14590.63205 , 25333.

In [18]:
age = float(input("Age: "))
bmi = float(input("BMI: "))
children = float(input("Children: "))
sex = float(input("Sex Male 0 or 1: "))
smoker = float(input("Smoker Yes 0 or 1: "))
future_prediction = grid.predict(
    [[age, bmi, children, sex, smoker]]
)
future_prediction

Age:  32
BMI:  7.5
Children:  1
Sex Male 0 or 1:  0
Smoker Yes 0 or 1:  0


array([15230.32405])